# CELA.PRO — Sovereign AI OS v2026 (Colab)
الواجهة العربية (Gradio) مع قائمة اختيار المزوّد (`Groq` / `Gemini` / `OpenRouter` / `Kimi` / `Ollama`).

### إعداد الأسرار في Colab Secrets (🔑)
- `GROQ_API_KEY` — النموذج الافتراضي: `openai/gpt-oss-20b`
- `GEMINI_API_KEY` — النموذج الافتراضي: `gemini-2.5-flash`
- `OPENROUTER_API_KEY` — النموذج الافتراضي: `openrouter/free`
- `KIMI_API_KEY` — النموذج الافتراضي: `moonshot-v1-8k`
- `GRADIO_USERNAME` و `GRADIO_PASSWORD` — لحماية الرابط العام (`share=True`) من استهلاك حصص API.

In [ ]:
!pip install -q gradio httpx

In [ ]:
import os

SECRET_KEYS = (
    "GROQ_API_KEY",
    "GEMINI_API_KEY",
    "OPENROUTER_API_KEY",
    "KIMI_API_KEY",
    "OLLAMA_API_KEY",
    "GRADIO_USERNAME",
    "GRADIO_PASSWORD",
    "CELA_DEFAULT_PROVIDER",
    "GROQ_MODEL",
    "GEMINI_MODEL",
    "OPENROUTER_MODEL",
    "KIMI_MODEL",
    "OLLAMA_MODEL",
    # تجاوز عناوين المزوّدين (مثل نقطة Modal الوسيطة لـ Kimi: …/v1)
    "GROQ_BASE_URL",
    "GEMINI_BASE_URL",
    "OPENROUTER_BASE_URL",
    "KIMI_BASE_URL",
    "OLLAMA_BASE_URL",
)

try:
    from google.colab import userdata
except ImportError:
    userdata = None

loaded = []
for key in SECRET_KEYS:
    val = None
    if userdata is not None:
        try:
            val = userdata.get(key)
        except Exception:
            val = None
    if val:
        os.environ[key] = str(val).strip()
        loaded.append(key)

print("تم تحميل المتغيرات من Colab Secrets:", loaded or "(لم يُضبط أي سر بعد)")
print(
    "ℹ️ قيم المفاتيح لا تُطبع؛ تُعرض الأسماء فقط. "
    "لربط Kimi بنقطة Modal الوسيطة اضبط KIMI_BASE_URL وKIMI_MODEL "
    "وKIMI_API_KEY بصيغة TOKEN_ID.TOKEN_SECRET."
)


In [ ]:
# مزامنة مصدر التطبيق من origin/main — لا استخدام صامتًا لنسخة قديمة من cela_pro_app.py
import hashlib
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/sayedelazameydesign-crypto/open_source-.git"
REPO_DIR = pathlib.Path("cela_repo")
APP_PATH = pathlib.Path("cela_pro_app.py")
SOURCE_APP = REPO_DIR / "cela_pro_app.py"
MISSING = "لا شيء"


def run_git(*args):
    return subprocess.run(["git", *args], capture_output=True, text=True, check=False)


def short_digest(path):
    if not path.exists():
        return MISSING
    return hashlib.sha256(path.read_bytes()).hexdigest()[:12]


def last_line(text):
    lines = [line.strip() for line in (text or "").splitlines() if line.strip()]
    return lines[-1] if lines else "سبب غير معروف"


if REPO_DIR.exists():
    run_git("-C", str(REPO_DIR), "fetch", "origin", "main")
    pulled = run_git("-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main")
    sync_note = (
        "تم التحديث إلى أحدث origin/main (--ff-only)."
        if pulled.returncode == 0
        else f"⚠️ تعذّر التحديث --ff-only: {last_line(pulled.stderr)} — المصدر قد يكون قديمًا."
    )
else:
    cloned = run_git("clone", REPO_URL, str(REPO_DIR))
    sync_note = (
        "تم استنساخ المستودع (origin/main)."
        if cloned.returncode == 0
        else f"⚠️ تعذّر الاستنساخ: {last_line(cloned.stderr)}"
    )

head = run_git("-C", str(REPO_DIR), "rev-parse", "--short", "HEAD")
source_sha = head.stdout.strip() if head.returncode == 0 else "غير معروف"

if SOURCE_APP.exists():
    before = short_digest(APP_PATH)
    if APP_PATH.exists() and before != short_digest(SOURCE_APP):
        backup = APP_PATH.with_name("cela_pro_app.local.bak.py")
        backup.write_bytes(APP_PATH.read_bytes())
        print(f"🗂️ نسختك المحلية مختلفة، حُفظت في {backup.name} قبل الاستبدال.")
    APP_PATH.write_text(SOURCE_APP.read_text(encoding="utf-8"), encoding="utf-8")
    after = short_digest(APP_PATH)
    if before == MISSING:
        print(f"✅ أُنشئ cela_pro_app.py من المستودع عند الالتزام {source_sha}.")
    elif before == after:
        print(f"✅ cela_pro_app.py مطابق لنسخة المستودع عند الالتزام {source_sha}.")
    else:
        print(f"🔄 استُبدل cela_pro_app.py بنسخة المستودع عند الالتزام {source_sha}.")
elif APP_PATH.exists():
    print(
        "⚠️ نسخة المستودع غير متاحة؛ سيُستخدم الملف المحلي "
        f"({short_digest(APP_PATH)}) وقد يكون قديمًا."
    )

if not APP_PATH.exists():
    try:
        from google.colab import files

        print("ارفع ملف cela_pro_app.py المرافق:")
        files.upload()
    except ImportError:
        pass

if "cela_pro_app" in sys.modules:
    print(
        "ℹ️ الوحدة مستوردة مسبقًا في هذه الجلسة: "
        "أعد تشغيل الخلايا التالية (أو importlib.reload) لتحميل النسخة الجديدة."
    )

print("حالة مزامنة المصدر:", sync_note)
print("التزام المصدر:", source_sha)
print(
    "حالة ملف التطبيق:",
    f"موجود ✅ ({short_digest(APP_PATH)})" if APP_PATH.exists() else "غير موجود ❌",
)


In [ ]:
# خلية الفحص الدقيق (Smoke Test) — تُنفَّذ قبل تشغيل واجهة Gradio
# ترسل طلب تحقق واحدًا إلى Groq و Gemini وتتأكد من وصول ردٍّ حقيقي
# يتضمن عبارة «اختبار الاتصال ناجح».
# الحالات: BLOCKED (المفتاح غير مُعيّن) · FAIL (فشل الطلب أو رسالة خطأ)
#          INCONCLUSIVE (رد بلا عبارة التحقق) · PASS (رد يتضمن عبارة التحقق)
# قيمة مفتاح API تُحجب دائمًا [REDACTED] في المخرجات، ولا تبديل تلقائيًا بين المزوّدين.
import cela_pro_app

SMOKE_PROVIDERS = ("Groq", "Gemini")

smoke_summary = {}
for provider_name in SMOKE_PROVIDERS:
    result = cela_pro_app.run_smoke_test(provider_name)
    smoke_summary[provider_name] = result["state"]
    print(f"=== فحص {provider_name} — الحالة: {result['state']} ===")
    print(cela_pro_app.format_smoke_result(result))
    print()

print("ملخص الفحص الدقيق:", smoke_summary)
if all(state == "PASS" for state in smoke_summary.values()):
    print("✅ جميع المزوّدين المختبَرين جاهزون — تابع تشغيل الواجهة.")
else:
    print(
        "⚠️ يوجد مزوّد غير جاهز: راجع التفصيل أعلاه "
        "(مفتاح ناقص أو فشل اتصال أو رد بلا عبارة التحقق)."
    )


In [ ]:
import cela_pro_app

auth = cela_pro_app.get_launch_auth()
if auth is None:
    print(
        "⚠️ تحذير: أضف GRADIO_USERNAME و GRADIO_PASSWORD في Colab Secrets "
        "لحماية الرابط العام من استهلاك حصص API."
    )

cela_pro_app.demo.launch(share=True, auth=auth, css=cela_pro_app.CSS)
